# 04 — Spatial Probability Map

For a chosen site and query date, pairs every `loc` tile with its healthy
reference (same `loc` ID, different date) and runs inference on each pair.

Produces two figures:
1. **Tile grid** — each loc tile shown as RGB with its P(bleached) score
   and a color-coded border (red = bleached, green = healthy)
2. **Geographic map** — tile footprints plotted at real coordinates
   colored by score (most useful when many tiles cover a reef)

**Data layout expected:**
```
<DATA_ROOT>/<SITE>/bleached/tiled_360m/<QUERY_DATE>/loc*.tif
<DATA_ROOT>/<SITE>/healthy/tiled_360m/<REF_DATE>/loc*.tif
```

In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
CKPT_PATH   = "/pscratch/sd/k/kevinval/coraltest/ct_classifier/model_states/best.pt"
PROJECT_DIR = "/pscratch/sd/k/kevinval/coraltest/ct_classifier"
DATA_ROOT   = "/pscratch/sd/k/kevinval/coraltest/ct_classifier/planet_superdove_landmasked"
FIGURES_DIR = "figures"

SITE        = "sanagustin_mexico"
QUERY_DATE  = "20230705"   # bleaching event date (YYYYMMDD)
REF_DATE    = "20210801"   # healthy reference date (YYYYMMDD)

In [ ]:
import os, sys, glob
import yaml
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib
import rasterio
import torch

if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

from ct_classifier.model import CustomResNet
from utils.viz import to_display_rgb, percentile_stretch

os.makedirs(FIGURES_DIR, exist_ok=True)

cfg_path = os.path.join(os.path.dirname(CKPT_PATH), "config.yaml")
with open(cfg_path) as f:
    cfg = yaml.safe_load(f)

device = "cuda" if torch.cuda.is_available() else "cpu"
norm_factor = float(cfg["normalization_factor"])

model = CustomResNet(cfg["num_classes"], cfg["layers"])
state = torch.load(CKPT_PATH, map_location="cpu")
model.load_state_dict(state["model"])
model.to(device).eval()
print("Model loaded. Device:", device)

In [ ]:
# ── Collect and match loc tiles ───────────────────────────────────────────────
query_dir = os.path.join(DATA_ROOT, SITE, "bleached", "tiled_360m", QUERY_DATE)
ref_dir   = os.path.join(DATA_ROOT, SITE, "healthy",  "tiled_360m", REF_DATE)

query_tiles = sorted(glob.glob(os.path.join(query_dir, "loc*.tif")))
ref_tiles   = sorted(glob.glob(os.path.join(ref_dir,   "loc*.tif")))

assert query_tiles, f"No query tiles found in:\n  {query_dir}"
assert ref_tiles,   f"No reference tiles found in:\n  {ref_dir}"

def loc_id(path):
    return os.path.splitext(os.path.basename(path))[0]  # "loc001", "loc002", ...

ref_by_loc = {loc_id(p): p for p in ref_tiles}
pairs = [(q, ref_by_loc[loc_id(q)]) for q in query_tiles if loc_id(q) in ref_by_loc]

print(f"Query tiles : {len(query_tiles)}")
print(f"Ref tiles   : {len(ref_tiles)}")
print(f"Matched pairs: {len(pairs)}")

In [ ]:
# ── Run inference on all pairs ────────────────────────────────────────────────
def read_tile(path):
    with rasterio.open(path) as src:
        arr    = src.read().astype(np.float32)
        bounds = src.bounds
    valid = np.isfinite(arr).all(axis=0).astype(np.float32)
    arr   = np.nan_to_num(arr, nan=0.0)
    return arr, valid, bounds

results = []
with torch.no_grad():
    for q_path, r_path in pairs:
        q_arr, q_mask, q_bounds = read_tile(q_path)
        r_arr, r_mask, _        = read_tile(r_path)

        x    = np.concatenate([r_arr, q_arr], axis=0) / norm_factor
        m    = np.stack([r_mask, q_mask], axis=0)
        tile = torch.from_numpy(
            np.concatenate([x, m], axis=0)
        ).unsqueeze(0).to(device)

        prob = torch.softmax(model(tile), dim=1)[0, 1].item()
        results.append({
            "loc":       loc_id(q_path),
            "prob":      prob,
            "bounds":    q_bounds,
            "query_arr": q_arr,
            "ref_arr":   r_arr,
        })
        print(f"  {loc_id(q_path)}  P(bleached)={prob:.3f}")

print(f"\nDone. {len(results)} tiles scored.")

In [ ]:
# ── Figure 1: Tile grid ───────────────────────────────────────────────────────
# Each loc tile shown as RGB (ref left, query right) with score + color border.
# Border color: red = high P(bleached), green = low.
cmap = matplotlib.colormaps["RdYlGn_r"]

n = len(results)
fig, axes = plt.subplots(n, 2, figsize=(7, n * 3.2))
if n == 1:
    axes = axes[np.newaxis, :]

fig.suptitle(
    f"{SITE}\nRef: {REF_DATE}  →  Query: {QUERY_DATE}",
    fontsize=12, y=1.01
)

for row, r in enumerate(results):
    border_color = cmap(r["prob"])

    ref_rgb   = percentile_stretch(to_display_rgb(torch.from_numpy(r["ref_arr"])))
    query_rgb = percentile_stretch(to_display_rgb(torch.from_numpy(r["query_arr"])))

    for col, (img, title) in enumerate([
        (ref_rgb,   f"{r['loc']} — REF ({REF_DATE})"),
        (query_rgb, f"{r['loc']} — QUERY ({QUERY_DATE})  P(bleached)={r['prob']:.2f}"),
    ]):
        axes[row, col].imshow(img)
        axes[row, col].set_title(title, fontsize=9)
        axes[row, col].axis("off")
        # color-coded border
        for spine in axes[row, col].spines.values():
            spine.set_visible(True)
            spine.set_edgecolor(border_color)
            spine.set_linewidth(5)

fig.tight_layout()
fig.savefig(
    os.path.join(FIGURES_DIR, f"prob_grid_{SITE}_{QUERY_DATE}.png"),
    dpi=150, bbox_inches="tight"
)
plt.show()

In [ ]:
# ── Figure 2: Geographic map ──────────────────────────────────────────────────
# Tile footprints at real coordinates colored by P(bleached).
# Most useful when many tiles cover a reef — shows spatial bleaching patterns.
fig, ax = plt.subplots(figsize=(8, 6))

for r in results:
    b     = r["bounds"]
    color = cmap(r["prob"])
    rect  = mpatches.FancyBboxPatch(
        (b.left, b.bottom),
        b.right - b.left, b.top - b.bottom,
        boxstyle="square,pad=0",
        linewidth=1, edgecolor="white",
        facecolor=color, alpha=0.9
    )
    ax.add_patch(rect)
    ax.text(
        (b.left + b.right) / 2, (b.bottom + b.top) / 2,
        f"{r['loc']}\n{r['prob']:.2f}",
        ha="center", va="center", fontsize=8,
        color="white", fontweight="bold"
    )

all_bounds = [r["bounds"] for r in results]
x_min = min(b.left   for b in all_bounds)
x_max = max(b.right  for b in all_bounds)
y_min = min(b.bottom for b in all_bounds)
y_max = max(b.top    for b in all_bounds)
pad   = max(x_max - x_min, y_max - y_min) * 0.15

ax.set_xlim(x_min - pad, x_max + pad)
ax.set_ylim(y_min - pad, y_max + pad)
ax.set_aspect("equal")
ax.set_xlabel("Easting (m)")
ax.set_ylabel("Northing (m)")
ax.set_title(f"{SITE} — {QUERY_DATE}\nBleaching probability by tile location")

sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(0, 1))
sm.set_array([])
plt.colorbar(sm, ax=ax, fraction=0.03, pad=0.04, label="P(bleached)")

fig.tight_layout()
fig.savefig(
    os.path.join(FIGURES_DIR, f"spatial_map_{SITE}_{QUERY_DATE}.png"),
    dpi=150, bbox_inches="tight"
)
plt.show()